# Preprocesamiento, Ventaneo y División del Dataset para MyoTensor Proto (S07)

Este notebook realiza el pipeline completo de procesamiento para el canal único de sEMG de **MyoTensor Proto** (S07):
1. Filtro de outliers (`valid_flag == 1`).
2. Segmentación contigua por bloques de estado de estímulo (evita mezclar tiempos no contiguos).
3. Extracción de ventanas deslizantes con solapamiento ($W=300$ ms, $S=100$ ms).
4. División en conjuntos de **Entrenamiento (80%)** y **Prueba (20%)** estratificados.
5. Exportación de tensores en formato NumPy binario (`.npy`), incluyendo etiquetas categóricas (para SVM/RF) y etiquetas one-hot (para tu red neuronal CNN-LSTM).

---

## ¿Qué Estructura Tienen los Tensores Finales?

### Conjunto de Entrenamiento (80%):
*   **`X_train.npy`:** Tensor 3D de shape `(1638, 300, 1)` (1,638 ventanas de 300 ms con 1 canal).
*   **`y_train.npy`:** Vector 1D de shape `(1638,)` con etiquetas enteras (`0, 1, 2, 3`) para SVM/Random Forest.
*   **`y_train_onehot.npy`:** Matriz 2D de shape `(1638, 4)` con codificación one-hot para tu red **CNN-LSTM**.

### Conjunto de Prueba/Validación (20%):
*   **`X_test.npy`:** Tensor 3D de shape `(410, 300, 1)` (410 ventanas de 300 ms con 1 canal).
*   **`y_test.npy`:** Vector 1D de shape `(410,)` con etiquetas enteras (`0, 1, 2, 3`) para SVM/Random Forest.
*   **`y_test_onehot.npy`:** Matriz 2D de shape `(410, 4)` con codificación one-hot para tu red **CNN-LSTM**.

## 1. Configuración de Parámetros Globales

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Función puramente robusta para buscar y cargar el archivo .env
def load_env_variables():
    import os
    from pathlib import Path
    
    # 1. Buscar .env subiendo niveles desde el CWD actual
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    # 2. Leer e inyectar variables en os.environ
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()


# ======================================================================
# PARÁMETROS CONFIGURABLES
# ======================================================================
W = 300         # Tamaño de la ventana (300 ms a 1000 Hz)
S = 100         # Paso de la ventana (100 ms para 66.6% de solapamiento)
TEST_SIZE = 0.2 # 20% para el conjunto de prueba (Test/Validation)
SEED = 42       # Semilla para que el split sea reproducible

# Carga de rutas desde las variables del .env
BASE_DIR = Path(os.environ["RAW_DATA_PROTO"])
OUTPUT_DIR = Path(os.environ["PROCESSED_TENSOR_PROTO"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ======================================================================

2026-05-24 23:21:07.733184: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-24 23:21:07.768387: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-24 23:21:08.552886: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


## 2. Algoritmo de Preprocesamiento y Extracción de Ventanas

In [2]:
X_list = []
y_list = []

# Buscar específicamente la sesión CSV de S07
csv_files = sorted(list(BASE_DIR.glob("S07/session_*.csv")))
print(f"[*] PASO 1: Buscando archivos CSV en {BASE_DIR}")
print(f"    -> Se encontró el archivo de S07: {[f.name for f in csv_files]}\n")

for csv_path in csv_files:
    print(f"[*] PASO 2: Cargando archivo {csv_path.name}...")
    df = pd.read_csv(csv_path)
    print(f"    -> Muestras totales en crudo: {len(df)}")
    
    # 1. Filtro de muestras de calidad valid_flag == 1
    df_valid = df[df["valid_flag"] == 1].copy()
    n_outliers = len(df) - len(df_valid)
    print(f"[*] PASO 3: Aplicando filtro de calidad (valid_flag == 1)")
    print(f"    -> Muestras válidas conservadas: {len(df_valid)} (outliers eliminados: {n_outliers})")
    
    # 2. Identificar bloques contiguos de restimulus (gestos o reposos)
    block_id = (df_valid["restimulus"] != df_valid["restimulus"].shift()).cumsum()
    groups = df_valid.groupby(block_id)
    
    print(f"[*] PASO 4: Segmentando señal en bloques contiguos de actividad/reposo...")
    print(f"    -> Se detectaron {len(groups)} bloques contiguos de señal en la sesión.\n")
    print(f"{'Bloque ID':<10} | {'Clase Gesto':<12} | {'Duración (muestras)':<20} | {'Ventanas Extraídas':<20}")
    print("-" * 70)
    
    block_count = 1
    gesture_names = {0: "Reposo (0)", 1: "Palma (1)", 2: "Puño (2)", 3: "Paz (3)"}
    
    for g_id, group_df in groups:
        gesture_class = group_df["restimulus"].iloc[0]
        class_name = gesture_names.get(gesture_class, f"Clase {gesture_class}")
        
        # Señal normalizada
        sig_col = "emg_norm" if "emg_norm" in group_df.columns else "filtered"
        signal_block = group_df[sig_col].values
        L = len(signal_block)
        
        # Si el bloque es más corto que la ventana configurable W, se descarta
        if L < W:
            print(f"Block {block_count:<5} | {class_name:<12} | {L:<20} | 0 (Descartado: L < W)")
            block_count += 1
            continue
            
        # 3. Ventaneo Deslizante
        extracted_in_block = 0
        for start in range(0, L - W + 1, S):
            window = signal_block[start:start + W]
            X_list.append(window)
            y_list.append(gesture_class)
            extracted_in_block += 1
            
        print(f"Block {block_count:<5} | {class_name:<12} | {L:<20} | {extracted_in_block:<20}")
        block_count += 1

print(f"\n[+] PASO 5: Ventaneo finalizado. Total de ventanas generadas: {len(X_list)}")

[*] PASO 1: Buscando archivos CSV en /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/datasets/raw/myotensor_proto
    -> Se encontró el archivo de S07: ['session_20260426_222602.csv']

[*] PASO 2: Cargando archivo session_20260426_222602.csv...
    -> Muestras totales en crudo: 242160
[*] PASO 3: Aplicando filtro de calidad (valid_flag == 1)
    -> Muestras válidas conservadas: 217963 (outliers eliminados: 24197)
[*] PASO 4: Segmentando señal en bloques contiguos de actividad/reposo...
    -> Se detectaron 52 bloques contiguos de señal en la sesión.

Bloque ID  | Clase Gesto  | Duración (muestras)  | Ventanas Extraídas  
----------------------------------------------------------------------
Block 1     | Reposo (0)   | 3340                 | 31                  
Block 2     | Puño (2)     | 4940                 | 47                  
Block 3     | Reposo (0)   | 2695                 | 24                  
Block 4     | Paz (3)      | 5366                 | 51                  
Block 5

## 3. División del Dataset (Train/Test Split) con Estratificación y Normalización

In [3]:
from sklearn.preprocessing import StandardScaler
import joblib

print("[*] PASO 6: Convirtiendo listas a arreglos NumPy...")
X = np.array(X_list)
y = np.array(y_list)

# Expandir dimensión de canal único sEMG: (num_ventanas, W, 1)
X = np.expand_dims(X, axis=-1)
y = y.astype(np.int32)

print(f"[*] PASO 7: Dividiendo en conjuntos de Entrenamiento y Prueba...")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    stratify=y, 
    random_state=SEED
)

# Ajustar Standard Scaler a nivel muestra
scaler = StandardScaler()
N_train = X_train_raw.shape[0]
N_test = X_test_raw.shape[0]

X_train_flat = X_train_raw.reshape(N_train * W, 1)
X_test_flat  = X_test_raw.reshape(N_test * W, 1)

scaler.fit(X_train_flat)

X_train = scaler.transform(X_train_flat).reshape(N_train, W, 1)
X_test  = scaler.transform(X_test_flat).reshape(N_test, W, 1)

print(f"    -> Conjunto de Entrenamiento (Shape): X_train={X_train.shape}, y_train={y_train.shape}")
print(f"    -> Conjunto de Prueba (Shape):        X_test={X_test.shape}, y_test={y_test.shape}")

[*] PASO 6: Convirtiendo listas a arreglos NumPy...
[*] PASO 7: Dividiendo en conjuntos de Entrenamiento y Prueba...
    -> Conjunto de Entrenamiento (Shape): X_train=(1638, 300, 1), y_train=(1638,)
    -> Conjunto de Prueba (Shape):        X_test=(410, 300, 1), y_test=(410,)


## 4. Codificación One-Hot para Redes Neuronales (Keras / TensorFlow)

In [4]:
print("[*] PASO 8: Creando codificación One-Hot...")
num_classes = len(np.unique(y))
y_train_onehot = to_categorical(y_train, num_classes=num_classes)
y_test_onehot = to_categorical(y_test, num_classes=num_classes)

print(f"    -> y_train (One-Hot Shape): {y_train_onehot.shape}")
print(f"    -> y_test (One-Hot Shape):  {y_test_onehot.shape}")

[*] PASO 8: Creando codificación One-Hot...
    -> y_train (One-Hot Shape): (1638, 4)
    -> y_test (One-Hot Shape):  (410, 4)


## 5. Guardando Tensores en Carpeta de Procesados

In [5]:
print(f"[*] PASO 9: Guardando todos los tensores binarios en {OUTPUT_DIR}...")

np.save(OUTPUT_DIR / 'X_train.npy', X_train)
np.save(OUTPUT_DIR / 'y_train.npy', y_train_onehot)
np.save(OUTPUT_DIR / 'X_test.npy', X_test)
np.save(OUTPUT_DIR / 'y_test.npy', y_test_onehot)

# Guardar scaler
joblib.dump(scaler, OUTPUT_DIR / 'std_scaler.bin')

print("\n[OK] ¡Guardado Completado con Éxito!")

[*] PASO 9: Guardando todos los tensores binarios en /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/datasets/processed/myotensor_proto/tensor...

[OK] ¡Guardado Completado con Éxito!
